<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/02_informative_slope_prior.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 2 — Informative slope prior

Keep the same Gaussian regression but deliberately make the prior on the daily deprivation effect much tighter. This makes prior influence visible rather than merely discussing it abstractly.

## Setup

This course pins PyMC, modular ArviZ, and Bambi for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1" \
    "bambi==0.21.0"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bambi as bmb
import pymc as pm
import arviz_base as azb
import arviz_stats as azs
import arviz_plots as azp

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("Bambi:", bmb.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

That zero point is scientifically meaningful, so every Bambi model in this sequence uses `center_predictors=False`. The `Intercept` prior is therefore a prior on baseline reaction time rather than reaction time at the average deprivation day.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

print(f"{sleep['Subject'].nunique()} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

# 2.1 A skeptical slope prior

What does the model imply if we strongly expect the daily effect of sleep deprivation to be near zero?

The only change from notebook 1 is $\beta\sim N(0,1)$ ms/day. On the uncentered parameterization the intercept still describes baseline reaction time.

In [ ]:
priors = {
    "Intercept": bmb.Prior("Normal", mu=250, sigma=100),
    "Days": bmb.Prior("Normal", mu=0, sigma=1),
    "sigma": bmb.Prior("Exponential", lam=0.02),
}
model = bmb.Model(
    "Reaction ~ Days", sleep, family="gaussian", priors=priors, center_predictors=False
)
model

In [ ]:
prior = model.prior_predictive(draws=500, random_seed=RANDOM_SEED)
azp.plot_ppc_dist(
    prior,
    group="prior_predictive",
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

# 2.2 Data versus prior

How far does the posterior move away from a tight slope prior centered at zero?

In [ ]:
idata = model.fit(
    draws=1000,
    tune=1500,
    chains=4,
    target_accept=0.90,
    random_seed=RANDOM_SEED,
)

print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))


In [ ]:
azs.summary(idata, var_names=["Intercept", "Days", "sigma"], ci_prob=0.90, ci_kind="hdi", round_to=2)

In [ ]:
azp.plot_trace_dist(idata, var_names=["Intercept", "Days", "sigma"]);

In [ ]:
bmb.interpret.plot_predictions(
    model,
    idata,
    conditional="Days",
    target="mean",
    prob=[0.50, 0.90],
)

# 2.3 Consequences of prior influence

Does the informative slope prior materially change the model’s predictions, or mainly its estimate of the deprivation slope?

In [ ]:
model.predict(
    idata,
    kind="response",
    inplace=True,
    random_seed=RANDOM_SEED,
)

azp.plot_ppc_dist(
    idata,
    var_names=["Reaction"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

In [ ]:
model.compute_log_likelihood(idata)
model.compute_log_prior(idata)

azs.psense_summary(idata, var_names=["Intercept", "Days", "sigma"])

azp.plot_psense_dist(
    idata,
    var_names=["Intercept", "Days", "sigma"],
    visuals={"dist": False},
);

# 2.4 Limits of prior tuning

Can changing the slope prior address the participant-to-participant structure visible in the data?